# NB06 — Clean SAM recovery | Baego + Luki
**Aim:** inspect archived automatic SAM1/SAM3 runs and compare saved crown polygons with reference crowns. **No training or inference. No historical outputs modified.** The historical six-ROI SAM3 F1 ≈ 0.490 is a reference, **not** automatically reproduced here.

Run blocks in order. Select archived outputs in Block 04 only after reviewing filenames and metadata.

In [ ]:
# 01 — Setup (read-only paths)
from pathlib import Path
from google.colab import drive
import json, re, warnings
import numpy as np, pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from shapely.geometry import box
from rasterio.windows import from_bounds
from rasterio.windows import transform as window_transform

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Musanga')
DATA = ROOT / 'data/Geo_Ref'
ARCHIVE = ROOT / 'KVO_work/KVO_06_UNet_guided_SAM3'
ARCHIVE_V2 = ROOT / 'KVO_work/KVO_06_v2_Baego_Luki'
INPUTS = {'Baego_RGB': DATA/'Baego_georef.tif', 'Baego_Kristof': DATA/'Annotations/Crown_Kristof.shp', 'Baego_Nadine': DATA/'Annotations/Crown_Nadine.shp', 'Baego_ROI': DATA/'Annotations/ROI_Crowns_Kvo.shp', 'Luki_RGB': DATA/'Luki.tif', 'Luki_reference': DATA/'Annotations/Tree_Annotation_Luki.gpkg'}
for name,p in INPUTS.items(): print(f'{name:18} {"OK" if p.exists() else "MISSING"}  {p}')
print('Historical folders:', [(str(p),p.exists()) for p in (ARCHIVE,ARCHIVE_V2)])

In [ ]:
# 02 — Reference crowns and area rule (metres squared)
MIN_CROWN_M2 = 5.0

def load_layer(path, target_crs, raster_bounds, layer=None):
    g = gpd.read_file(path, layer=layer) if layer else gpd.read_file(path)
    g = g.loc[g.geometry.notna() & ~g.geometry.is_empty].copy()
    if g.empty:
        print(f'WARNING: no geometries in {path.name}')
        return g

    # A missing .prj is common in older annotation shapefiles.
    # Assign raster CRS only when the raw coordinate extents already overlap.
    if g.crs is None:
        xmin, ymin, xmax, ymax = g.total_bounds
        rxmin, rymin, rxmax, rymax = tuple(raster_bounds)
        overlaps = xmax >= rxmin and xmin <= rxmax and ymax >= rymin and ymin <= rymax
        if not overlaps:
            raise ValueError(
                f'{path.name} has no CRS and its coordinates do not overlap the RGB raster. '
                f'Annotation bounds={g.total_bounds}; raster bounds={tuple(raster_bounds)}. '
                'Check the original annotation CRS before proceeding.'
            )
        print(f'WARNING: {path.name} has no CRS; coordinates overlap raster. '
              f'Assuming raster CRS {target_crs}.')
        g = g.set_crs(target_crs, allow_override=True)
    else:
        g = g.to_crs(target_crs)
    return g

def load_crowns(path, target_crs, raster_bounds, layer=None):
    g = load_layer(path, target_crs, raster_bounds, layer)
    if not g.crs.is_projected:
        raise ValueError('A projected metric CRS is required for the 5 m² crown filter')
    if g.crs.axis_info and not all(
        axis.unit_name.lower() in ('metre','meter','metres','meters') for axis in g.crs.axis_info[:2]
    ):
        raise ValueError(f'CRS units are not metres: {g.crs}')
    before = len(g)
    g = g.loc[g.geometry.area >= MIN_CROWN_M2].copy()
    print(f'  {path.name}: {before} polygons → {len(g)} crowns ≥ {MIN_CROWN_M2:g} m²')
    return g

SITES = {}
for site, rgb_key in [('Baego','Baego_RGB'),('Luki','Luki_RGB')]:
    if not INPUTS[rgb_key].exists(): continue
    with rasterio.open(INPUTS[rgb_key]) as ds:
        if ds.crs is None or not ds.crs.is_projected:
            raise ValueError(f'{site}: projected raster CRS required')
        site_info = {'rgb':INPUTS[rgb_key], 'crs':ds.crs,
                     'bounds':ds.bounds, 'res':ds.res}
    crs, bounds = site_info['crs'], site_info['bounds']
    print(f'\n{site}: raster CRS={crs}, resolution={site_info["res"]}')
    if site == 'Baego':
        site_info['Kristof'] = load_crowns(INPUTS['Baego_Kristof'], crs, bounds)
        site_info['Nadine'] = load_crowns(INPUTS['Baego_Nadine'], crs, bounds)
        site_info['roi'] = load_layer(INPUTS['Baego_ROI'], crs, bounds)
    else:
        site_info['reference'] = load_crowns(
            INPUTS['Luki_reference'], crs, bounds,
            layer='photointerpretation_inventory')
    SITES[site] = site_info
    print('Reference counts:', {
        k:len(v) for k,v in site_info.items()
        if isinstance(v,gpd.GeoDataFrame)
    })

In [ ]:
# 03 — Inspect archive files + original SAM configuration (do not execute archived code)
# This is a text inspection, not model initialization.
SEARCH_ROOTS = [p for p in (ROOT/'KVO_work',ROOT) if p.exists()]
SEARCH_ROOT = SEARCH_ROOTS[0]
notebooks = sorted(SEARCH_ROOT.rglob('*.ipynb'))
print('SAM notebooks (up to 60):')
for p in [p for p in notebooks if any(x in p.name.lower() for x in ('sam','06','crown'))][:60]: print('  ',p.relative_to(ROOT))
print('\nLikely historical configs:')
PAT = re.compile(r'(points_per_batch|points_per_side|points_per_crop|crops_n_layers|crop_overlap_ratio|pred_iou_thresh|stability_score_thresh|stability_score_offset|mask_threshold|max_hole_area|max_sprinkle_area|tile_size|tile_px|overlap|context|sam1|sam3)',re.I)
for p in [p for p in notebooks if ('sam' in p.name.lower() or '06' in p.name.lower())][:30]:
    try:
        obj=json.loads(p.read_text(errors='replace'))
        hits=[]
        for i,c in enumerate(obj.get('cells',[])):
            src=''.join(c.get('source',[]))
            for line in src.splitlines():
                if PAT.search(line) and len(line)<220 and not line.lstrip().startswith('#'):
                    hits.append((i,line.strip()))
        if hits:
            print('\n',p.relative_to(ROOT))
            for i,line in hits[:25]: print(f'  cell {i}: {line[:180]}')
    except Exception as e: print('Cannot inspect',p.name,str(e)[:100])

In [ ]:
# 04 — Find existing saved polygons; manually select a verified historical run
# No file is written. Do not select watershed, YOLO, RF, or fine-tuned outputs.
FOLDERS = [p for p in (ARCHIVE,ARCHIVE_V2,ROOT/'KVO_work/KVO_06_SAM3_Baego_Crown_Discovery') if p.exists()]
for folder in FOLDERS:
    print('\n',folder.relative_to(ROOT))
    candidates = [p for p in folder.rglob('*') if p.suffix.lower() in ('.gpkg','.geojson','.shp','.csv','.png','.json') and any(x in str(p).lower() for x in ('sam1','sam3','amg','historical','baseline','v53','v56','v58'))]
    for p in candidates[:100]: print(' ',p.relative_to(ROOT))
    if len(candidates)>100: print(' ...',len(candidates)-100,'more')

# Set only after checking paths and actual layer content. Leave None to skip overlays.
PREDICTION_PATHS = {'Baego': None, 'Luki': None}
PREDICTION_LAYERS = {'Baego': None, 'Luki': None}
PREDICTIONS = {}
for site,p in PREDICTION_PATHS.items():
    if p is None: continue
    p=Path(p)
    if not p.exists(): raise FileNotFoundError(p)
    g=gpd.read_file(p,layer=PREDICTION_LAYERS[site]) if PREDICTION_LAYERS[site] else gpd.read_file(p)
    if g.crs is None: raise ValueError(f'Missing prediction CRS: {p}')
    PREDICTIONS[site]=g.to_crs(SITES[site]['crs'])
    print(site,'selected historical polygons:',len(PREDICTIONS[site]))

In [ ]:
# 05 — Visual RGB + reference + saved SAM polygons; no inference
# Adjust BOUNDS per site to inspect a specific historical ROI. None uses first ROI for Baego and full extent for Luki.
VIEW_BOUNDS={'Baego':None,'Luki':None}  # (xmin,ymin,xmax,ymax), in that site's raster CRS

def display_site(site):
    s=SITES[site]
    with rasterio.open(s['rgb']) as ds:
        if VIEW_BOUNDS[site] is not None:
            bb=VIEW_BOUNDS[site]
        elif site=='Baego' and len(s['roi']):
            bb=s['roi'].geometry.iloc[0].bounds
        else:
            # Small central preview instead of trying to render a huge full orthomosaic
            cx=(ds.bounds.left+ds.bounds.right)/2; cy=(ds.bounds.bottom+ds.bounds.top)/2
            half=min(ds.bounds.right-ds.bounds.left,ds.bounds.top-ds.bounds.bottom)/16
            bb=(cx-half,cy-half,cx+half,cy+half)
        w=from_bounds(*bb,transform=ds.transform).round_offsets().round_lengths()
        from rasterio.windows import Window
        w=w.intersection(Window(0,0,ds.width,ds.height))
        if w.width<=0 or w.height<=0: raise ValueError('Window is outside the image')
        rgb=ds.read([1,2,3],window=w,out_shape=(3,min(int(w.height),1500),min(int(w.width),1500)))
        bounds=rasterio.windows.bounds(w,ds.transform)
    image=np.moveaxis(rgb,0,-1).astype(float)
    lo,hi=np.nanpercentile(image,[2,98]); image=np.clip((image-lo)/(hi-lo+1e-9),0,1)
    fig,ax=plt.subplots(figsize=(10,10));ax.imshow(image,extent=[bounds[0],bounds[2],bounds[1],bounds[3]])
    roi_box=box(*bounds)
    refkeys=['Kristof','Nadine'] if site=='Baego' else ['reference']
    for key,color in zip(refkeys,['white','yellow']):
        g=s[key]; g=g.loc[g.geometry.intersects(roi_box)]
        if len(g):g.boundary.plot(ax=ax,color=color,linewidth=.9,label=key)
    if site in PREDICTIONS:
        g=PREDICTIONS[site];g=g.loc[g.geometry.intersects(roi_box)]
        if len(g):g.boundary.plot(ax=ax,color='red',linewidth=1.0,label='Historical SAM')
    ax.set_title(f'{site} | reference (white/yellow) | saved SAM (red)');ax.set_aspect('equal');plt.show()

for site in SITES: display_site(site)

In [ ]:
# 06 — Baseline status + optional evaluation table
# Avoid claiming that a new matching implementation reproduces the historical F1.
HISTORICAL_SIX_ROI_F1_APPROX = 0.490
print(f'Historical six-ROI SAM3 benchmark: F1 ≈ {HISTORICAL_SIX_ROI_F1_APPROX:.3f} (reported, not recomputed)')
print('Selected archived predictions:',{k:len(v) for k,v in PREDICTIONS.items()})
print('Before comparing metrics, retrieve original six ROI IDs and original matching/IoU and postprocessing rules from notebook/config.')
print('Reference area filter:',MIN_CROWN_M2,'m²')
print('No training, inference, or output writing occurred.')

## 07 — Decision log (fill after inspecting original archive)
- **Original notebook and run ID:** not yet verified
- **SAM1 checkpoint / SAM3 model revision:** not yet verified
- **Tiling dimensions / overlap / context:** not yet verified
- **Prompting mode:** not yet verified
- **AMG thresholds:** not yet verified (do not confuse Hugging Face defaults with actual run settings)
- **Postprocessing / overlap handling:** not yet verified
- **Six historical ROI identifiers and saved output:** not yet verified
- **Historical SAM3 F1 ≈ 0.490:** preserved as reported benchmark only

**Decision gate:** only consider a new inference block after the archived run and its visual outputs have been positively identified.